In [ ]:
import pandas as pd
from data_loader import load_train_data

train_source1, train_source2, train_source3, ground_truth = load_train_data(
    "../../../dataset"
)

In [29]:
import pandas as pd

train_source1 = pd.read_csv("/Users/aaravraj/Downloads/student_resource/dataset/train/train_source1.tsv",sep="\t")

train_source2 = pd.read_csv("/Users/aaravraj/Downloads/student_resource/dataset/train/train_source2.tsv",sep="\t")

train_source3 = pd.read_csv("/Users/aaravraj/Downloads/student_resource/dataset/train/train_source3.tsv",sep="\t")

ground_truth = pd.read_csv("/Users/aaravraj/Downloads/student_resource/dataset/train/train_ground_truth.tsv",sep="\t")

In [31]:
train_source1.info()
train_source2.info()
train_source3.info()
ground_truth.info()

<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 2206820
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 259.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 5034616 entries, 0 to 5034615
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 602.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 5285603 entries, 0 to 5285602
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 622.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 

In [33]:
print(train_source1.shape)
print(train_source2.shape)
print(train_source3.shape)
print(ground_truth.shape)

(2206821, 4)
(5034616, 4)
(5285603, 4)
(2206821, 2)


In [35]:
for name, df in {
    "Source 1": train_source1,
    "Source 2": train_source2,
    "Source 3": train_source3
}.items():
    
    print("\n", name)
    print(df.isnull().sum())


 Source 1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

 Source 2
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

 Source 3
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64


In [37]:
for df in [train_source1, train_source2, train_source3]:
    print((df == "").sum())

entity_id           0
business_name       0
business_address    0
country             0
dtype: int64
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64


In [39]:
print("Source 1 duplicates:",
      train_source1.duplicated().sum())

print("Source 2 duplicates:",
      train_source2.duplicated().sum())

print("Source 3 duplicates:",
      train_source3.duplicated().sum())

Source 1 duplicates: 0
Source 2 duplicates: 0
Source 3 duplicates: 0


In [41]:
for df in [train_source1, train_source2, train_source3]:
    print(df["entity_id"].duplicated().sum())

0
0
0


In [43]:
import re
import unicodedata

def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)
    
    # Remove accents
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    
    # Replace & with and
    text = text.replace("&", " and ")
    
    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [45]:
for df in [train_source1, train_source2, train_source3]:

    df["business_name_clean"] = (
        df["business_name"]
        .apply(normalize_text)
    )

In [47]:
def remove_legal_suffixes(text):
    suffixes = [
        "private limited",
        "pvt limited",
        "pvt ltd",
        "private ltd",
        "limited",
        "ltd",
        "llc",
        "inc",
        "incorporated",
        "corporation",
        "corp"
    ]
    
    text = text.lower()
    
    for suffix in suffixes:
        text = re.sub(
            rf"\b{re.escape(suffix)}\b",
            "",
            text
        )
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [49]:
for df in [train_source1, train_source2, train_source3]:

    df["business_name_core"] = (
        df["business_name_clean"]
        .apply(remove_legal_suffixes)
    )

In [51]:
def normalize_address(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    text = unicodedata.normalize("NFKD", text)
    
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    
    text = text.replace("&", " and ")
    
    # Common address abbreviations
    replacements = {
        r"\brd\b": "road",
        r"\brd\.\b": "road",
        r"\bst\b": "street",
        r"\bst\.\b": "street",
        r"\bave\b": "avenue",
        r"\bav\b": "avenue",
        r"\bblvd\b": "boulevard",
        r"\bln\b": "lane",
        r"\bdr\b": "drive",
        r"\bhwy\b": "highway",
    }
    
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    
    # punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [53]:
for df in [train_source1, train_source2, train_source3]:

    df["business_address_clean"] = (
        df["business_address"]
        .apply(normalize_address)
    )

In [57]:
def normalize_country(text):
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    
    return text

In [59]:
for df in [train_source1, train_source2, train_source3]:

    df["country_clean"] = (
        df["country"]
        .apply(normalize_country)
    )

In [61]:
def tokenize(text):
    return set(text.split()) 

In [63]:
for df in [train_source1, train_source2, train_source3]:

    df["name_tokens"] = (
        df["business_name_clean"]
        .apply(tokenize)
    )

In [ ]:
def extract_pincode(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Indian PIN
    match = re.search(r"\b\d{6}\b", text)
    
    if match:
        return match.group()
    
    # Generic 5 digit ZIP
    match = re.search(r"\b\d{5}\b", text)
    
    if match:
        return match.group()
    
    return ""

: 

In [ ]:
for df in [train_source1, train_source2, train_source3]:

    df["pincode"] = (
        df["business_address"]
        .apply(extract_pincode)
    )

: 